# Notebook Overview — Train Two Classifiers

## Purpose

This notebook performs **final model validation and training** for the Digital Image Processing (DIP)–based AI image detection pipeline.

It re-evaluates the selected classifiers using **stratified cross-validation**, then trains the finalized models on the full training dataset and saves both the trained models and their configurations for downstream evaluation.

---

## Inputs

Required inputs:

* Normalized feature vector CSV file:
  * `metadata/vectors/<train_feature_vectors_normalized_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Normalized training feature vectors are loaded
* Dataset structure, metadata, and feature dimensionality are validated
* Feature matrix (`X_train`) and target vector (`y_train`) are constructed
* Class labels are encoded into numeric form
* Final classifier configurations (MLP and RBF SVM) are defined
* Stratified cross-validation is performed on selected classifiers
* Classifier performance is summarized and ranked
* Final classifiers are trained using the full training dataset
* Trained models are serialized and saved
* Cross-validation results are saved to CSV
* Model configuration metadata is saved to JSON
* Processing is deterministic and reproducible

---

## Outputs

**Cross-Validation Results**  
`metadata/results/<cross_validation_results_filename>`

**Trained Models**  
`metadata/models/<final_mlp_model_filename>`  
`metadata/models/<final_rbf_svm_model_filename>`

**Model Configuration Files**  
`metadata/models/<best_mlp_model_config_filename>`  
`metadata/models/<best_rbf_svm_model_config_filename>`

Each configuration file contains:

* Classifier type
* Hyperparameters
* Number of features (26)
* Cross-validation settings
* Training sample count
* Selected evaluation metric (ROC AUC)
* Cross-validation performance (mean ROC AUC)
* Path to saved model file

---

## Expected Behavior

* Training dataset passes all validation checks
* Feature dimensionality remains consistent (26 features)
* Cross-validation produces stable and reproducible results
* Selected classifiers (MLP and RBF SVM) perform consistently across folds
* Final models are successfully trained on the full dataset
* Trained models are saved without error
* Cross-validation results and configuration files are correctly persisted
* Outputs are ready for independent test evaluation in subsequent notebooks

---
---

### 🔷 Step 1 — Startup: Environment and Verification

- Import required libraries for final model training and cross-validation
- Configure notebook display behavior using `VERBOSE`
- Optionally suppress warnings for cleaner output
- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input path for normalized training feature vectors
- Define output paths for final trained models, model configurations, and cross-validation results
- Create required model and results directories if they do not exist
- Verify that required normalized training data is available
- Optionally display configuration paths when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Startup — Environment and Verification
# ============================================

# ============================================
# USER CONFIGURATION — EDIT THIS SECTION ONLY
# ============================================

VERBOSE = True   # User toggle (True or False)

# ============================================
# END USER CONFIGURATION
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
import json
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# -------------------------------------------------
# Suppress warnings for cleaner output (optional)
# -------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_PATH,
    NUM_FEATURES,
    RANDOM_SEED,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    K_FOLDS,
    CV_SHUFFLE,
    CV_RANDOM_STATE,
    MODELS_METADATA_DIR,
    RESULTS_METADATA_DIR,
    FINAL_MLP_MODEL_PATH,
    FINAL_RBF_SVM_MODEL_PATH,
    BEST_MLP_MODEL_CONFIG_PATH,
    BEST_RBF_SVM_MODEL_CONFIG_PATH,
    CROSS_VALIDATION_RESULTS_PATH,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = Path(TRAIN_NORMALIZED_PATH)

MODELS_DIR = Path(MODELS_METADATA_DIR)
RESULTS_DIR = Path(RESULTS_METADATA_DIR)

FINAL_MLP_MODEL_PATH = Path(FINAL_MLP_MODEL_PATH)
FINAL_RBF_SVM_MODEL_PATH = Path(FINAL_RBF_SVM_MODEL_PATH)

BEST_MLP_MODEL_CONFIG_PATH = Path(BEST_MLP_MODEL_CONFIG_PATH)
BEST_RBF_SVM_MODEL_CONFIG_PATH = Path(BEST_RBF_SVM_MODEL_CONFIG_PATH)

CV_RESULTS_CSV = Path(CROSS_VALIDATION_RESULTS_PATH)

# -------------------------------------------------
# Ensure required output directories exist
# -------------------------------------------------
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    missing_files.append(str(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV))

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

print("All required input files are present.")

# -------------------------------------------------
# Optional verbose display of configuration paths
# -------------------------------------------------
if VERBOSE:
    print(f"Training data     : {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"Models directory  : {MODELS_DIR}")
    print(f"Results directory : {RESULTS_DIR}")
    print(f"CV results CSV    : {CV_RESULTS_CSV}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nLibraries imported successfully.")



Cloning repository...
Verifying required input files...

All required input files are present.
Training data     : /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Models directory  : /content/dip-ai-image-detection/metadata/models
Results directory : /content/dip-ai-image-detection/metadata/results
CV results CSV    : /content/dip-ai-image-detection/metadata/results/cross_validation_results.csv

Libraries imported successfully.


### 🔷 Step 2 — Load Normalized Training Data

- Define path to normalized training feature vector CSV file
- Load normalized training dataset into a DataFrame
- Confirm successful data loading
- Optionally display dataset shape when `VERBOSE=True`
- Optionally preview first few rows of the dataset

---

In [ ]:
# ============================================
# Step 2: Load Normalized Training Data
# ============================================

# -------------------------------------------------
# Define input CSV path
# -------------------------------------------------
train_csv_path = TRAIN_FEATURE_VECTORS_NORMALIZED_CSV

# -------------------------------------------------
# Load normalized training dataset
# -------------------------------------------------
df_train = pd.read_csv(train_csv_path)

print("Normalized training feature table loaded successfully.\n")

# -------------------------------------------------
# Optional verbose display of dataset info
# -------------------------------------------------
if VERBOSE:
    print(f"Training CSV: {train_csv_path}\n")
    print(f"df_train shape: {df_train.shape}\n")

# -------------------------------------------------
# Optional preview of dataset
# -------------------------------------------------
if VERBOSE:
    print("First 5 rows of training table:")
    try:
        display(df_train.head())
    except:
        print(df_train.head())



Normalized training feature table loaded successfully.

Training CSV: /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv

df_train shape: (14400, 30)

First 5 rows of training table:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Noise Residual Energy,Low Frequency Energy Ratio,Mid Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.071297,-0.189219,0.111086,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,-0.330229,-0.264290,0.292101,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.851489,-0.544033,0.442499,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.188882,0.518813,-0.552523,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.281104,0.607047,-0.600533,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


### 🔷 Step 3 — Validate Training Data and Prepare Features/Labels

- Verify presence of required metadata columns
- Identify feature columns and validate feature count
- Ensure no missing values are present in the dataset
- Separate feature matrix `X_train` and target vector `y_train`
- Validate class labels against expected values (`ai`, `real`)
- Encode class labels into numeric format using `LabelEncoder`
- Optionally display label mapping, class distribution, and data shapes when `VERBOSE=True`
- Confirm dataset is valid and ready for model training

---

In [ ]:
# ============================================
# Step 3: Validate Training Data and Prepare Features/Labels
# ============================================

print("Performing sanity checks and preparing training data...\n")

# -------------------------------------------------
# Define expected metadata columns
# -------------------------------------------------
expected_metadata_columns = METADATA_COLUMNS.copy()

# -------------------------------------------------
# Verify required metadata columns are present
# -------------------------------------------------
missing_train_metadata = [
    col for col in expected_metadata_columns
    if col not in df_train.columns
]

if missing_train_metadata:
    raise ValueError(
        f"Training table is missing metadata columns: {missing_train_metadata}"
    )

if VERBOSE:
    print("Required metadata columns verified.")

# -------------------------------------------------
# Identify and validate feature columns
# -------------------------------------------------
feature_columns = [
    col for col in df_train.columns
    if col not in expected_metadata_columns
]

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, "
        f"but found {len(feature_columns)}."
    )

if VERBOSE:
    print(f"Feature count verified: {len(feature_columns)}")

# -------------------------------------------------
# Check for missing values
# -------------------------------------------------
if df_train.isnull().sum().sum() != 0:
    raise ValueError("Training table contains missing values.")

if VERBOSE:
    print("No missing values detected.")

# -------------------------------------------------
# Separate feature matrix (X) and target vector (y)
# -------------------------------------------------
X_train = df_train[feature_columns].copy()
y_train = df_train["class_label"].copy()

# -------------------------------------------------
# Validate label values
# -------------------------------------------------
expected_labels = {AI_LABEL, REAL_LABEL}
train_labels_found = set(y_train.unique())

if not train_labels_found.issubset(expected_labels):
    raise ValueError(
        f"Unexpected labels found in training set: {train_labels_found}"
    )

if VERBOSE:
    print(f"Label values verified: {sorted(expected_labels)}")

# -------------------------------------------------
# Encode class labels to numeric format
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# -------------------------------------------------
# Optional verbose display of dataset properties
# -------------------------------------------------
if VERBOSE:
    print("\nLabel encoding mapping:")
    for i, label in enumerate(label_encoder.classes_):
        print(f"{label} -> {i}")

    print("\nTraining class counts:")
    print(y_train.value_counts())

    print("\nFeature matrix shape:")
    print(f"X_train: {X_train.shape}")

    print("\nLabel vector shapes:")
    print(f"y_train:         {y_train.shape}")
    print(f"y_train_encoded: {y_train_encoded.shape}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nSanity checks passed successfully.")



Performing sanity checks and preparing training data...

Required metadata columns verified.
Feature count verified: 26
No missing values detected.
Label values verified: ['ai', 'rl']

Label encoding mapping:
ai -> 0
rl -> 1

Training class counts:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Feature matrix shape:
X_train: (14400, 26)

Label vector shapes:
y_train:         (14400,)
y_train_encoded: (14400,)

Sanity checks passed successfully.


### 🔷 Step 4 — Define Selected Classifier Configurations

- Define final classifier configurations selected from previous model selection notebook
- Include top-performing classifiers (MLP and RBF SVM)
- Specify tuned hyperparameters for each classifier
- Prepare configurations for cross-validation and final training
- Optionally display classifier configurations when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 4: Define Selected Classifier Configurations
# ============================================

print("Defining selected classifier configurations...\n")

# -------------------------------------------------
# Define final classifier configurations
# (based on results from Notebook 07)
# -------------------------------------------------
classifier_configs = [
    {
        "classifier": "MLP",
        "params": {
            "hidden_layer_sizes": [64, 32],
            "alpha": 0.0001,
            "learning_rate_init": 0.001,
            "batch_size": 32
        }
    },
    {
        "classifier": "RBF SVM",
        "params": {
            "C": 100,
            "gamma": 0.01
        }
    }
]

# -------------------------------------------------
# Optional verbose display of configurations
# -------------------------------------------------
if VERBOSE:
    print(f"Number of classifiers configured: {len(classifier_configs)}\n")

    for i, config in enumerate(classifier_configs, 1):
        print(f"{i}. {config['classifier']}")
        print("   Parameters:")
        for param, value in config["params"].items():
            print(f"     {param} = {value}")
        print()



Defining selected classifier configurations...

Number of classifiers configured: 2

1. MLP
   Parameters:
     hidden_layer_sizes = [64, 32]
     alpha = 0.0001
     learning_rate_init = 0.001
     batch_size = 32

2. RBF SVM
   Parameters:
     C = 100
     gamma = 0.01



### 🔷 Step 5 — Cross-Validate Selected Classifiers

- Define stratified cross-validation strategy using configured fold settings
- Define evaluation metrics for selected classifiers
- Instantiate each selected classifier from its saved configuration
- Run cross-validation for MLP and RBF SVM models
- Compute mean and standard deviation for accuracy, precision, recall, F1 score, and ROC AUC
- Store classifier parameters with cross-validation results
- Track runtime for each model and the full evaluation
- Optionally display progress and timing details when `VERBOSE=True`
- Confirm cross-validation is complete

---

In [ ]:
# ============================================
# Step 5: Cross-Validate Selected Classifiers
# ============================================

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc",
}

# -------------------------------------------------
# Initialize results storage
# -------------------------------------------------
results = []

print("Evaluating selected classifiers...\n")

# -------------------------------------------------
# Optional verbose display of evaluation setup
# -------------------------------------------------
if VERBOSE:
    print(f"Number of classifiers: {len(classifier_configs)}")
    print(f"Cross-validation folds: {K_FOLDS}\n")

total_start_time = time.time()

# -------------------------------------------------
# Evaluate each selected classifier
# -------------------------------------------------
for i, config in enumerate(classifier_configs, start=1):
    model_start_time = time.time()

    classifier_name = config["classifier"]
    params = config["params"].copy()

    print(f"[{i}/{len(classifier_configs)}] {classifier_name}")

    # -------------------------------------------------
    # Instantiate model from configuration
    # -------------------------------------------------
    if classifier_name == "MLP":
        if "hidden_layer_sizes" in params:
            params["hidden_layer_sizes"] = tuple(params["hidden_layer_sizes"])
        params["max_iter"] = 300
        params["random_state"] = RANDOM_SEED
        model = MLPClassifier(**params)

    elif classifier_name == "RBF SVM":
        params["kernel"] = "rbf"
        params["probability"] = True
        params["random_state"] = RANDOM_SEED
        model = SVC(**params)

    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # -------------------------------------------------
    # Run cross-validation
    # -------------------------------------------------
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train_encoded,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # -------------------------------------------------
    # Store cross-validation summary row
    # -------------------------------------------------
    row = {
        "classifier": classifier_name,
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std": cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std": cv_results["test_recall"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std": cv_results["test_f1"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

    # -------------------------------------------------
    # Store classifier parameters with result row
    # -------------------------------------------------
    for param_name, param_value in params.items():
        row[param_name] = str(param_value)

    results.append(row)

    # -------------------------------------------------
    # Track runtime
    # -------------------------------------------------
    model_elapsed = time.time() - model_start_time
    total_elapsed = time.time() - total_start_time

    if VERBOSE:
        print(f"  Completed in {model_elapsed:.2f} seconds")
        print(f"  Total elapsed: {total_elapsed:.2f} seconds\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Cross-validation complete.")

if VERBOSE:
    print(f"Total runtime: {time.time() - total_start_time:.2f} seconds")



Evaluating selected classifiers...

Number of classifiers: 2
Cross-validation folds: 5

[1/2] MLP
  Completed in 49.43 seconds
  Total elapsed: 49.43 seconds

[2/2] RBF SVM
  Completed in 44.03 seconds
  Total elapsed: 93.46 seconds

Cross-validation complete.
Total runtime: 93.46 seconds


### 🔷 Step 6 — Summarize and Rank Classifier Results

- Convert cross-validation results into a DataFrame
- Verify that cross-validation results are available
- Sort classifiers by mean ROC AUC score
- Optionally display full ranked comparison table when `VERBOSE=True`
- Display ranked summary of classifier performance
- Select top classifiers to retain for final training
- Report ROC AUC, F1 score, and accuracy for retained classifiers

---

In [ ]:
# ============================================
# Step 6: Summarize and Rank Classifier Results
# ============================================

# -------------------------------------------------
# Convert cross-validation results to DataFrame
# -------------------------------------------------
df_cv_results = pd.DataFrame(results)

# -------------------------------------------------
# Validate cross-validation results are available
# -------------------------------------------------
if df_cv_results.empty:
    raise ValueError("No cross-validation results found.")

# -------------------------------------------------
# Sort results by primary metric (ROC-AUC)
# -------------------------------------------------
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print("Classifier comparison summarized successfully.")

# -------------------------------------------------
# Optional verbose display of ranked results
# -------------------------------------------------
if VERBOSE:
    print("\nClassifier comparison (sorted by ROC-AUC):\n")

    try:
        display(df_cv_results)
    except:
        print(df_cv_results)

    print("\nRanked classifier summary:\n")

    for i, row in df_cv_results.iterrows():
        print(
            f"{i+1}. {row['classifier']} | "
            f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
            f"F1: {row['f1_mean']:.4f} | "
            f"Accuracy: {row['accuracy_mean']:.4f}"
        )

# -------------------------------------------------
# Identify top classifiers retained for final training
# -------------------------------------------------
TOP_K_FINAL = min(2, len(df_cv_results))
top_rows = df_cv_results.head(TOP_K_FINAL)

print(f"\nTop {TOP_K_FINAL} classifiers retained for final training:\n")

for i, (_, row) in enumerate(top_rows.iterrows(), 1):
    print(f"{i}. {row['classifier']}")
    print(f"   ROC-AUC:  {row['roc_auc_mean']:.4f}")
    print(f"   F1 Score: {row['f1_mean']:.4f}")
    print(f"   Accuracy: {row['accuracy_mean']:.4f}")
    print()



Classifier comparison summarized successfully.

Classifier comparison (sorted by ROC-AUC):



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,...,hidden_layer_sizes,alpha,learning_rate_init,batch_size,max_iter,random_state,C,gamma,kernel,probability
0,RBF SVM,0.798889,0.008192,0.799013,0.008107,0.798889,0.008192,0.798867,0.008208,0.879407,...,NaN,NaN,NaN,NaN,NaN,42,100,0.01,rbf,True
1,MLP,0.764931,0.007915,0.765830,0.006905,0.764931,0.007915,0.764717,0.008169,0.844270,...,"(64, 32)",0.0001,0.001,32,300,42,NaN,NaN,NaN,NaN



Ranked classifier summary:

1. RBF SVM | ROC-AUC: 0.8794 | F1: 0.7989 | Accuracy: 0.7989
2. MLP | ROC-AUC: 0.8443 | F1: 0.7647 | Accuracy: 0.7649

Top 2 classifiers retained for final training:

1. RBF SVM
   ROC-AUC:  0.8794
   F1 Score: 0.7989
   Accuracy: 0.7989

2. MLP
   ROC-AUC:  0.8443
   F1 Score: 0.7647
   Accuracy: 0.7649



### 🔷 Step 7 — Train Final Models on Full Training Set

- Initialize storage for trained classifier models
- Instantiate each selected classifier using tuned configurations
- Train each model using the full normalized training dataset
- Store trained models for later evaluation and saving
- Optionally display training progress when `VERBOSE=True`
- Confirm all models have been successfully trained

---

In [ ]:
# ============================================
# Step 7: Train Final Models on Full Training Set
# ============================================

# -------------------------------------------------
# Initialize storage for trained models
# -------------------------------------------------
trained_models = {}

print("Training final models on full training dataset...\n")

# -------------------------------------------------
# Train each selected classifier using full dataset
# -------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"].copy()

    print(f"Training: {classifier_name}")

    # -------------------------------------------------
    # Instantiate model from configuration
    # -------------------------------------------------
    if classifier_name == "MLP":
        if "hidden_layer_sizes" in params:
            params["hidden_layer_sizes"] = tuple(params["hidden_layer_sizes"])
        params["max_iter"] = 300
        params["random_state"] = RANDOM_SEED
        model = MLPClassifier(**params)

    elif classifier_name == "RBF SVM":
        params["kernel"] = "rbf"
        params["probability"] = True
        params["random_state"] = RANDOM_SEED
        model = SVC(**params)

    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # -------------------------------------------------
    # Train model on full training dataset
    # -------------------------------------------------
    model.fit(X_train, y_train_encoded)

    # -------------------------------------------------
    # Store trained model
    # -------------------------------------------------
    trained_models[classifier_name] = model

    if VERBOSE:
        print(f"  Completed: {classifier_name}\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("All models trained successfully.")



Training final models on full training dataset...

Training: MLP
  Completed: MLP

Training: RBF SVM
  Completed: RBF SVM

All models trained successfully.


### 🔷 Step 8 — Save Trained Models

- Initialize storage for saved model file paths
- Assign output file paths based on classifier type
- Serialize trained models using `pickle`
- Save trained models to disk
- Record saved model paths for later use
- Optionally display saved model paths when `VERBOSE=True`
- Confirm all trained models are saved successfully

---

In [ ]:
# ============================================
# Step 8: Save Trained Models
# ============================================

# -------------------------------------------------
# Initialize storage for saved model paths
# -------------------------------------------------
saved_model_paths = {}

print("Saving trained models...\n")

# -------------------------------------------------
# Save each trained model to disk
# -------------------------------------------------
for model_name, model in trained_models.items():

    # -------------------------------------------------
    # Select output path based on model type
    # -------------------------------------------------
    if model_name == "MLP":
        model_path = FINAL_MLP_MODEL_PATH
    elif model_name == "RBF SVM":
        model_path = FINAL_RBF_SVM_MODEL_PATH
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    # -------------------------------------------------
    # Serialize and save model
    # -------------------------------------------------
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    # -------------------------------------------------
    # Record saved path
    # -------------------------------------------------
    saved_model_paths[model_name] = model_path

    if VERBOSE:
        print(f"Saved: {model_name}")
        print(f"  Path: {model_path}\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("All models saved successfully.")



Saving trained models...

Saved: MLP
  Path: /content/dip-ai-image-detection/metadata/models/final_mlp_model.pkl

Saved: RBF SVM
  Path: /content/dip-ai-image-detection/metadata/models/final_rbf_svm_model.pkl

All models saved successfully.


### 🔷 Step 9 — Save Cross-Validation Summary and Model Configurations

- Save cross-validation results to a CSV file
- Verify successful persistence of evaluation results
- Build configuration dictionaries for each trained classifier
- Include:
  - Classifier type
  - Hyperparameters
  - Feature count
  - Cross-validation settings
  - Performance metrics
  - Model file path
- Save each model configuration as a JSON file
- Optionally display saved file paths when `VERBOSE=True`
- Confirm all model configurations are saved successfully

---

In [ ]:
# ============================================
# Step 9: Save Cross-Validation Summary and Model Configs
# ============================================

print("Saving cross-validation summary and model configurations...\n")

# -------------------------------------------------
# Save cross-validation summary to CSV
# -------------------------------------------------
df_cv_results.to_csv(CV_RESULTS_CSV, index=False)

print("Cross-validation summary saved successfully.")

if VERBOSE:
    print(f"CSV: {CV_RESULTS_CSV}")

# -------------------------------------------------
# Save per-model configuration files (JSON)
# -------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"]

    # -------------------------------------------------
    # Retrieve matching evaluation row
    # -------------------------------------------------
    matching_row = df_cv_results[
        df_cv_results["classifier"] == classifier_name
    ].iloc[0]

    # -------------------------------------------------
    # Build model configuration dictionary
    # -------------------------------------------------
    model_config = {
        "classifier": classifier_name,
        "hyperparameters": params,
        "num_features": NUM_FEATURES,
        "k_folds": K_FOLDS,
        "train_samples": int(len(X_train)),
        "selected_metric": "roc_auc",
        "roc_auc_mean": float(matching_row["roc_auc_mean"]),
        "model_path": str(saved_model_paths[classifier_name]),
    }

    # -------------------------------------------------
    # Select output path based on classifier
    # -------------------------------------------------
    if classifier_name == "MLP":
        output_path = BEST_MLP_MODEL_CONFIG_PATH
    elif classifier_name == "RBF SVM":
        output_path = BEST_RBF_SVM_MODEL_CONFIG_PATH
    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # -------------------------------------------------
    # Save configuration to JSON
    # -------------------------------------------------
    with open(output_path, "w") as f:
        json.dump(model_config, f, indent=4)

    if VERBOSE:
        print(f"Saved config for {classifier_name}:")
        print(f"  JSON: {output_path}\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("All model configurations saved successfully.")



Saving cross-validation summary and model configurations...

Cross-validation summary saved successfully.
CSV: /content/dip-ai-image-detection/metadata/results/cross_validation_results.csv
Saved config for MLP:
  JSON: /content/dip-ai-image-detection/metadata/models/best_mlp_model_config.json

Saved config for RBF SVM:
  JSON: /content/dip-ai-image-detection/metadata/models/best_rbf_svm_model_config.json

All model configurations saved successfully.
